<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>02. Sparse and Scalable Gaussian Processes</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [The Scalability Challenge](#-part-i-scalability-challenge)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Inducing Points and Sparse GPs](#-part-iii-sparse-gps)**
    - 3.1 [The Inducing Point Framework](#inducing-point-framework)
    - 3.2 [Variational Sparse GP with TFP](#variational-sparse-gp)
- **4. [Scaling to Large Datasets](#-part-iv-scaling)**
- **5. [Key Takeaways](#-part-v-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. The Scalability Challenge</span>

| **N (data points)** | **Exact GP** | **Sparse GP (M inducing)** |
|:---:|:---:|:---:|
| 100 | ✅ 0.001s | ✅ 0.001s |
| 1,000 | ✅ 0.1s | ✅ 0.01s |
| 10,000 | ⚠️ 10s | ✅ 0.5s |
| 100,000 | ❌ Infeasible | ✅ 5s |
| 1,000,000 | ❌ Infeasible | ✅ 50s |

- Exact GP: $\mathcal{O}(N^3)$ time, $\mathcal{O}(N^2)$ memory
- Sparse GP: $\mathcal{O}(NM^2)$ time, $\mathcal{O}(NM)$ memory
- With mini-batching: $\mathcal{O}(BM^2)$ per step — **truly scalable!**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
import time

tfd = tfp.distributions
tfk = tfp.math.psd_kernels

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Inducing Points and Sparse GPs</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. The Inducing Point Framework</span>

Instead of using all $N$ training points, we select $M \ll N$ **inducing points** $Z = \{z_1, \ldots, z_M\}$:

$$p(f | X, Z) \approx q(f) = \int p(f | u, X, Z) q(u) \, du$$

Where $u = f(Z)$ are the function values at inducing points and $q(u) = \mathcal{N}(m, S)$ is a variational distribution.

The variational parameters $\{Z, m, S\}$ are optimized jointly with kernel hyperparameters by maximizing the **ELBO**.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Variational Sparse GP with TFP</span>

In [ ]:
# Generate a large dataset
np.random.seed(42)
N = 2000
X_train = np.sort(np.random.uniform(-5, 5, N)).astype(np.float32).reshape(-1, 1)
y_train = (np.sin(X_train[:, 0]) * np.cos(0.5 * X_train[:, 0]) + 
           0.2 * np.random.randn(N)).astype(np.float32)

print(f"Training data: {N} points")

# Exact GP for comparison (on subset)
subset_size = 200
idx = np.random.choice(N, subset_size, replace=False)
X_sub, y_sub = X_train[idx], y_train[idx]

# Sparse GP with inducing points
M = 30  # Number of inducing points
inducing_points = tf.Variable(
    np.linspace(-5, 5, M).reshape(-1, 1).astype(np.float32),
    name='inducing_points'
)

# Trainable kernel
log_amp = tf.Variable(0.0, name='log_amp')
log_ls = tf.Variable(0.0, name='log_ls')
log_noise = tf.Variable(-2.0, name='log_noise')

def build_sparse_gp():
    kernel = tfk.ExponentiatedQuadratic(
        amplitude=tf.exp(log_amp),
        length_scale=tf.exp(log_ls)
    )
    return tfd.VariationalGaussianProcess(
        kernel=kernel,
        index_points=X_train,
        inducing_index_points=inducing_points,
        variational_inducing_observations_loc=tf.Variable(
            tf.zeros(M), name='var_loc'
        ),
        variational_inducing_observations_scale=tf.linalg.LinearOperatorDiag(
            tf.nn.softplus(tf.Variable(tf.zeros(M), name='var_scale'))
        ),
        observation_noise_variance=tf.exp(log_noise)
    )

print(f"Sparse GP with {M} inducing points (compression: {N/M:.0f}x)")
print(f"Exact GP would require {N}x{N} = {N*N:,} matrix operations")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Scaling to Large Datasets</span>

In [ ]:
# Compare exact vs sparse GP on the subset
kernel_exact = tfk.ExponentiatedQuadratic(amplitude=1.0, length_scale=0.5)
X_test = np.linspace(-5.5, 5.5, 300).astype(np.float32).reshape(-1, 1)

# Exact GP
start = time.time()
gprm = tfd.GaussianProcessRegressionModel(
    kernel=kernel_exact,
    index_points=X_test,
    observation_index_points=X_sub.reshape(-1, 1),
    observations=y_sub,
    observation_noise_variance=0.04
)
exact_mean = gprm.mean().numpy()
exact_std = gprm.stddev().numpy()
exact_time = time.time() - start

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Exact GP
axes[0].fill_between(X_test[:, 0], exact_mean - 2*exact_std, exact_mean + 2*exact_std,
                     alpha=0.2, color='#3b82f6')
axes[0].plot(X_test[:, 0], exact_mean, color='#3b82f6', linewidth=2)
axes[0].scatter(X_sub[:, 0] if X_sub.ndim == 1 else X_sub, y_sub, c='#1e293b', s=10, alpha=0.5)
axes[0].set_title(f'Exact GP (n={subset_size}, {exact_time:.3f}s)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('x', fontsize=13)
axes[0].set_ylabel('f(x)', fontsize=13)

# Data for sparse GP panel
axes[1].scatter(X_train[:, 0], y_train, c='#94a3b8', s=3, alpha=0.3, label=f'Data (n={N})')
axes[1].scatter(inducing_points.numpy()[:, 0], np.zeros(M), c='#ef4444', 
                marker='^', s=100, zorder=5, label=f'Inducing pts (M={M})', edgecolors='white')
axes[1].set_title(f'Sparse GP Setup ({N} points, {M} inducing)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('x', fontsize=13)
axes[1].set_ylabel('f(x)', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Exact GP on {subset_size} points: {exact_time:.3f}s")
print(f"Full dataset ({N} points) would require ~{(N/subset_size)**3 * exact_time:.1f}s for exact GP")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| Exact GP | $O(N^3)$ — limited to ~10K points |
| Sparse GP | $O(NM^2)$ with $M$ inducing points |
| Inducing points | Learned pseudo-data that summarize the training set |
| Variational GP | Optimize ELBO for kernel params + inducing points |
| Mini-batch training | Enables truly large-scale GP inference |
| TFP class | `VariationalGaussianProcess` for sparse inference |

### 🎯 Practical Recommendations
- **N < 5,000**: Use exact GPs
- **5K < N < 100K**: Use sparse GPs with M = 100–500
- **N > 100K**: Use sparse GPs + mini-batching or deep kernel learning